# 07v1 - A4: submission pipeline (live decode + ensembled A2 v1 inference)

**Plan item A4** (see
docs/superpowers/specs/2026-08-27-a4-submission-pipeline-design.md).
Self-contained per this project's Kaggle constraint (no `import src`) -
`annotate_series_headers`/`normalise_laterality` below are hand-kept
copies of `src/preprocess.py`'s graduated functions; keep them in sync
manually if either side changes.

**Why this notebook exists:** this is a Code Competition - the hidden
test set only exists during the scored rerun (`<=9h`, internet
disabled), so the A3 pre-built pixel cache (built before the test set
existed) cannot cover it. This notebook decodes DICOM live instead,
using the same slot-recovery/pixel-decode/laterality logic the cache was
built with (ported from `data/raw/_reference_kernels/rsna-knee-500gb-to-11gib-cpu-pixel-cache.ipynb`),
so its output matches the cache's convention exactly - validated in
Section 2 below before trusting it on real hidden data.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F

RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
assert RAW_DIR.exists(), f"Competition data not found at {RAW_DIR}"

# The A3 train cache, same mount as 05v2/06v2 - needed here ONLY for the
# Section 2 validation gate (comparing live decode against known-correct
# pixels), never for the hidden test set itself.
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")
assert CACHE_DIR.exists(), f"A3 cache not found at {CACHE_DIR}"

# The 4 A2 v1 fold checkpoints, each its own separately-uploaded Kaggle
# Dataset (same convention 06v2 already used for folds 0-2 - fold 3 gets
# one too here). EDIT these paths to match the real slugs once attached
# (check `!ls /kaggle/input` if unsure - Kaggle slugifies the Dataset
# name you gave it, don't assume).
CHECKPOINT_PATHS = {
    0: Path("/kaggle/input/a2-v1-fold0-checkpoint/a2_v1_fold0_best.pt"),
    1: Path("/kaggle/input/a2-v1-fold1-checkpoint/a2_v1_fold1_best.pt"),
    2: Path("/kaggle/input/a2-v1-fold2-checkpoint/a2_v1_fold2_best.pt"),
    3: Path("/kaggle/input/a2-v1-fold3-checkpoint/a2_v1_fold3_best.pt"),
}
for _fold_id, _path in CHECKPOINT_PATHS.items():
    print(f"CHECKPOINT_PATHS[{_fold_id}]:", _path, "exists:", _path.exists())
    assert _path.exists(), f"fold {_fold_id} checkpoint not found at {_path}"

# Same values as config.py / models/a2_v1_checkpoint_metadata.json -
# hardcoded directly (matching 05v2/06v2's own convention) now that the
# checkpoints are 4 separate Dataset mounts rather than one bundle with
# its own metadata.json.
FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())
CV_FOLDS = 4
# The whole centre-anchor shortcut below (decode_study_slots decodes the
# window centre directly; the Section 2 gate diffs against cache[:, 3:6])
# is only valid for the middle group - not a free parameter, A2 v1 was
# only ever trained with group_index=1.
GROUP_INDEX = 1

SLOT_NAMES = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
CROP_MM = 130.0
IMG_SIZE = 224
GROUP_SIZE = 3
# Anchors per slot the source cache decodes (config.SLOT_CACHE_N_GROUPS).
# decode_study_slots below must decode all N_ANCHOR*GROUP_SIZE=9 slices
# and normalize over that full pool, matching read_slot(n_slice=9, ...)
# in the source - normalizing over fewer slices silently rescales
# brightness/contrast even when the selected pixels are identical.
N_ANCHOR = 3
# The exact window the real A3 cache was built with (cache_meta.json,
# RESOURCES.md) - NOT the source's per-plane default, which this run
# overrode. Must match exactly for the Section 2 gate to pass.
WINDOW = (0.35, 0.65)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

## Section 1: live decode - header probe, slot recovery, pixel crop, laterality

Order matters here and is the source's own: **probe headers first**
(`train_series.csv`/`test_series.csv` carry only the plane and two
coarse flags - the fat-suppression/weighting classification and the
laterality resolution both read DICOM header fields those CSVs do not
have), then classify slots, then resolve laterality **once per study**
(not per series - see `resolve_study_laterality`), then decode pixels.

In [ ]:
import re

_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
    r"water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")
FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}

# The header fields the slot classification and the laterality resolution
# below actually need. NONE of these are in train_series.csv /
# test_series.csv (which carry only StudyInstanceUID, SeriesInstanceUID,
# Fluid_Sensitive, Fat_Suppression, Anatomical_Plane), so they have to be
# read back off the DICOM headers - exactly what the source's own probe()
# does before annotate(). Feeding annotate_series_headers() raw CSV rows
# instead classifies every series as fatsat=False/weight=UNK/fluid=False
# and collapses the 6-slot recovery to at most 2 wrong slots.
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Laterality fallbacks: `Laterality` is absent on about half the
            # series, and 4 of the 12 targets are side-specific.
            "ImageLaterality", "StudyDescription", "BodyPartExamined",
            "PatientPosition", "ImagePositionPatient"]


def probe_series_headers(series_dir, study_id, series_uid):
    """One series' header fields, off its MIDDLE slice - port of
    stevenleehans's probe(), minus the corpus-wide threading.

    Middle slice rather than first: localiser/scout frames tend to sit at
    the ends of a stack. `stop_before_pixels=True` keeps this cheap (one
    small header read per series, not per slice).
    """
    row = {"StudyInstanceUID": study_id, "SeriesInstanceUID": series_uid,
           "dir": str(series_dir)}
    try:
        files = sorted(p.name for p in series_dir.glob("*.dcm"))
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(series_dir / files[len(files) // 2],
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def probe_study_series(raw_dir, split_dir, study_id, study_series_df):
    """Merge the DICOM header fields onto one study's *_series.csv rows.

    Must run BEFORE annotate_series_headers() and before laterality
    resolution - both read fields that only exist in the headers. Also
    supplies `n_slices` (the slot tie-break), so no separate glob pass is
    needed.
    """
    rows = [
        probe_series_headers(raw_dir / split_dir / study_id / uid, study_id, uid)
        for uid in study_series_df["SeriesInstanceUID"]
    ]
    probed = pd.DataFrame(rows).drop(columns=["StudyInstanceUID"])
    return study_series_df.merge(probed, on="SeriesInstanceUID", how="left")


def annotate_series_headers(series_df):
    """Hand-kept copy of src/preprocess.py::annotate_series_headers."""
    df = series_df.copy()
    for col in ("SeriesDescription", "SequenceName", "ScanOptions",
                "RepetitionTime", "EchoTime", "ScanningSequence"):
        if col not in df.columns:
            df[col] = None

    desc = df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna("")
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1 = desc.str.contains(_T1_RX)
    t2 = desc.str.contains(_T2_RX)
    pdw = desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    return df


def select_slot_series(study_series_df, plane_map):
    """One series per named slot for one study - port of stevenleehans's
    pick_slots(), scoped to a single study's rows.

    Ties broken toward the stack with the most slices (denser sampling).
    T1 slots (scarcest) fall back to any non-fat-sat series in-plane if
    the exact fluid=False match is missing.
    """
    g = study_series_df.copy()
    g["plane"] = g["SeriesInstanceUID"].map(plane_map)
    chosen = {}
    for name, plane, fluid, fs in SLOTS:
        sel = (g["plane"] == plane) & (g["fatsat"] == fs)
        if fluid is not None:
            sel = sel & (g["fluid"] == fluid)
        cand = g[sel]
        if len(cand) == 0 and fluid is False:
            cand = g[(g["plane"] == plane) & (~g["fatsat"])]
        if len(cand):
            chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
    return chosen


def normalise_laterality(img, plane, laterality):
    """Hand-kept copy of src/preprocess.py::normalise_laterality."""
    if laterality != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


# Word-boundary side markers, ported verbatim from the source. This
# corpus's reports/descriptions run to 9 languages, so an English-only
# \b(right|rt)\b / \b(left|lt)\b pair silently leaves most non-English
# studies unresolved. Deliberately NOT including bare Turkish "sag"
# (right): every sagittal protocol here is named `sag_...`, which would
# mark most of the dataset as right knees - same family of trap as GE's
# SAT_GEMS. The diacritic form is unambiguous and is matched.
_SIDE_RX = [
    ("R", re.compile(r"\b(right|rt|r_?knee|knee_?r|dexter|sağ|derech[ao]|rechts?|"
                     r"droite?|δεξ\w*)\b", re.I)),
    ("L", re.compile(r"\b(left|lt|l_?knee|knee_?l|sinister|sol|izquierd[ao]|links?|"
                     r"gauche|αριστερ\w*)\b", re.I)),
]


def _lat_from_text(*fields):
    """Side from any free-text header field, or None if absent OR
    contradictory - port of the source's _lat_from_text(). Two sides
    matching means the text is unreliable, so it resolves to nothing
    rather than to whichever regex happened to be checked first."""
    blob = " ".join(str(f) for f in fields if f and str(f).lower() != "nan")
    blob = re.sub(r"[^\w\s]", " ", blob)
    hits = {side for side, rx in _SIDE_RX if rx.search(blob)}
    return hits.pop() if len(hits) == 1 else None


def resolve_study_laterality(probed_df):
    """One study -> 'L' / 'R' / 'unknown', from ALL of its probed series.

    Port of the source's resolve_laterality(g), which is called once per
    StudyInstanceUID groupby group - NOT per series. Scope matters: the
    `Laterality` tag is missing on roughly half the series, so resolving
    per-series would give a study whose tag exists on only one series a
    MIXED convention across its own 6 slots (some flipped, some not),
    which is worse than not flipping at all.

    Evidence order: explicit Laterality tag -> ImageLaterality ->
    side words across SeriesDescription + StudyDescription +
    BodyPartExamined. The geometric (ImagePositionPatient x-sign)
    fallback is deliberately left out - the source's own default has it
    OFF (CFG.lat_from_geometry=0), and the cache this pipeline must match
    was built with it off.
    """
    def col(name):
        return probed_df.get(name, pd.Series(dtype=object))

    vals = [str(x).strip().upper() for x in col("Laterality").dropna()]
    vals = [v[0] for v in vals if v and v[0] in ("L", "R")]
    if vals:
        return vals[0]

    ivals = [str(x).strip().upper() for x in col("ImageLaterality").dropna()]
    ivals = [v[0] for v in ivals if v and v[0] in ("L", "R")]
    if ivals:
        return ivals[0]

    txt = _lat_from_text(*col("SeriesDescription").tolist(),
                         *col("StudyDescription").tolist(),
                         *col("BodyPartExamined").tolist())
    if txt:
        return txt
    return "unknown"


def geometric_slice_order(files):
    """Port of the source's order_slices() - project
    ImagePositionPatient onto the ImageOrientationPatient normal, falling
    back to SliceLocation, then InstanceNumber, then input order.

    DELIBERATELY NOT the same as src/data.py::geometric_slice_order().
    Ours here is ALL-OR-NOTHING per series, like the source: geometry is
    used only if EVERY file in the series has a position tag, otherwise
    the whole series falls through wholesale to the next method.
    src/data.py's version instead picks its fallback key per-FILE, which
    in degenerate series mixes geometry / SliceLocation / InstanceNumber
    across files of one series and yields a different order.

    Do not "fix" this into matching src/data.py: the A3 pixel cache these
    slots are validated against (Section 2) was built with the
    all-or-nothing chain, and matching it exactly is the point.
    """
    if len(files) < 2:
        return list(files)
    positions, locations, instances = [], [], []
    normal = None
    for f in files:
        ds = pydicom.dcmread(f, stop_before_pixels=True, force=True)
        orient = getattr(ds, "ImageOrientationPatient", None)
        if normal is None and orient is not None and len(orient) == 6:
            row_dir = np.array([float(v) for v in orient[:3]])
            col_dir = np.array([float(v) for v in orient[3:]])
            normal = np.cross(row_dir, col_dir)
        pos = getattr(ds, "ImagePositionPatient", None)
        positions.append(np.array([float(v) for v in pos]) if pos is not None else None)
        loc = getattr(ds, "SliceLocation", None)
        locations.append(float(loc) if loc is not None else None)
        num = getattr(ds, "InstanceNumber", None)
        instances.append(float(num) if num is not None else None)

    if normal is not None and all(p is not None for p in positions):
        return [f for _, f in sorted(zip((float(p @ normal) for p in positions), files))]
    if all(v is not None for v in locations):
        return [f for _, f in sorted(zip(locations, files))]
    if all(v is not None for v in instances):
        return [f for _, f in sorted(zip(instances, files))]
    return list(files)


def decode_study_slots(raw_dir, split_dir, study_id, chosen_series, laterality):
    """Decode one study's 6 named slots into (6, 3, IMG_SIZE, IMG_SIZE)
    uint8 + a (6,) presence mask - the live-decode equivalent of a row
    from the A3 pre-built cache.

    `laterality` is resolved ONCE PER STUDY by
    resolve_study_laterality() and passed in, matching the source (which
    builds a study -> side `lat_map` before decoding anything). Every
    slot of a study therefore gets the same flip convention.

    Decodes ALL N_ANCHOR*GROUP_SIZE=9 slices per slot (all 3 anchors),
    exactly like the source's read_slot(rec, n_slice=9, ...) - NOT just
    the centre anchor. An earlier version of this function decoded only
    the centre anchor's 3 slices directly, reasoning that
    np.linspace(lo, hi, 3)'s middle anchor equals (lo+hi)//2 for
    non-negative integer bounds - true for WHICH pixels get selected,
    but the source's [1, 99] percentile normalization is computed over
    the FULL 9-slice pooled volume before splitting into groups, not
    per 3-slice group. Normalizing over only 3 slices silently rescales
    brightness/contrast even when the selected pixels are identical -
    confirmed 2026-08-27 on real Kaggle data (Section 2 gate: one slot
    showed mean|diff|=58.6/255 against visually identical, correctly-
    positioned anatomy - a normalization-only divergence, not a wrong
    slice/series/flip). Only the centre group (GROUP_INDEX) is kept in
    the returned array; the other 2 anchors are decoded solely to get
    the normalization statistics right, then discarded - the same
    computation the cache did, just not all of it persisted.
    """
    cache = np.zeros((len(SLOT_NAMES), GROUP_SIZE, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    mask = np.zeros(len(SLOT_NAMES), dtype=np.float32)
    n_slice = N_ANCHOR * GROUP_SIZE

    for k, (name, plane, _, _) in enumerate(SLOTS):
        if name not in chosen_series:
            continue
        row = chosen_series[name]
        series_dir = raw_dir / split_dir / study_id / row["SeriesInstanceUID"]
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            continue
        files = geometric_slice_order(files)
        n = len(files)

        lo_f, hi_f = WINDOW
        lo, hi = int(lo_f * (n - 1)), int(hi_f * (n - 1))
        if hi <= lo:
            lo, hi = 0, n - 1

        anchors = (np.linspace(lo, hi, N_ANCHOR).astype(int) if N_ANCHOR > 1
                   else np.array([(lo + hi) // 2]))
        idx = []
        for anchor_centre in anchors:
            start = int(np.clip(anchor_centre - GROUP_SIZE // 2, 0, max(0, n - GROUP_SIZE)))
            idx.extend(range(start, min(start + GROUP_SIZE, n)))
        while len(idx) < n_slice:
            idx.append(idx[-1])
        idx = idx[:n_slice]

        planes, px = [], None
        for i in idx:
            # Per-slice try/except, like the source: one unreadable slice
            # must degrade to a black plane, not take the whole study's 6
            # slots down with it (an un-caught np.stack ValueError here
            # propagates all the way out to the per-study handler).
            try:
                ds = pydicom.dcmread(files[i], force=True)
                if px is None:
                    spacing = getattr(ds, "PixelSpacing", None)
                    px = float(spacing[0]) if spacing else None
                a = ds.pixel_array.astype(np.float32)
                sl = float(getattr(ds, "RescaleSlope", 1) or 1)
                ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
                a = a * sl + ic
                if str(getattr(ds, "PhotometricInterpretation", "")).strip() == "MONOCHROME1":
                    a = a.max() - a
            except Exception:
                a = None
            planes.append(a)

        shp = next((p.shape for p in planes if p is not None), None)
        if shp is None:
            continue  # every slice failed - leave the slot absent, mask stays 0
        planes = [p if (p is not None and p.shape == shp) else np.zeros(shp, np.float32)
                  for p in planes]
        vol = np.stack(planes)  # (9, h, w)

        if px and np.isfinite(px) and px > 0:
            want = int(round(CROP_MM / px))
            h, w = shp
            if 16 < want < min(h, w):
                cy, cx = h // 2, w // 2
                half = want // 2
                vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

        # Percentile normalization over the FULL 9-slice pooled volume -
        # must match the source's scope exactly (see docstring above).
        lo_v, hi_v = np.percentile(vol, [1, 99])
        vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)
        t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
        t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        t = t.squeeze(0)  # (9, IMG_SIZE, IMG_SIZE)

        # Keep only the centre group now that normalization has already
        # used the full 9-slice pool - matches take_group(rows, g=1).
        group_start = GROUP_INDEX * GROUP_SIZE
        t = t[group_start:group_start + GROUP_SIZE]
        t = normalise_laterality(t, plane, laterality)

        cache[k] = (t * 255).round().clamp(0, 255).to(torch.uint8).numpy()
        mask[k] = 1.0

    return cache, mask


print("Live decode functions defined.")

## Section 2: validation gate - live decode vs. the pre-built train cache

Picks a handful of real gold **train** studies (DICOM available on
Kaggle same as always) and diffs `decode_study_slots()`'s output against
those same studies' already-known-correct pixels in the pre-built A3
cache. This is the only way to gain confidence before trusting the live
decode on real hidden test data, which offers no such cross-check.

In [ ]:
train_csv = pd.read_csv(RAW_DIR / "train.csv")
train_series_csv = pd.read_csv(RAW_DIR / "train_series.csv")
gold_mask = train_csv[LABEL_COLS].notna().all(axis=1)
gold_study_ids = train_csv.loc[gold_mask, "StudyInstanceUID"].tolist()
print(f"{len(gold_study_ids)} gold studies available for the validation gate")

plane_map = dict(zip(train_series_csv["SeriesInstanceUID"], train_series_csv["Anatomical_Plane"]))

# Load shard 0 of the pre-built cache (memory-mapped, cheap) to diff against.
cache_studies = pd.read_csv(CACHE_DIR / "train.s00of04_studies.csv")
cache_arr = np.load(CACHE_DIR / "train.s00of04_cache.npy", mmap_mode="r")
cache_row_of = {sid: i for i, sid in enumerate(cache_studies["StudyInstanceUID"])}

# Pick the check studies AFTER the shard is loaded, from the studies it
# actually holds - shard 0 is only ~1/4 of the corpus, so taking
# gold_study_ids[:5] blind would make the overlap a coin flip and could
# fail the n_compared assert below on a perfectly healthy pipeline.
# ALL of the overlap, not just 5: after the normalization-pool fix
# (2026-08-27) found and fixed a real bug that a 5-study sample caught,
# one single study still diverges on one slot with no evidence of a
# code bug on inspection (clean per-slice DICOM headers, correct
# anchor/crop/percentile math) - widened to see whether that is a rare,
# isolated case or a real remaining pattern before deciding how to treat it.
CHECK_STUDIES = [s for s in gold_study_ids if s in cache_row_of]
print(f"{len(CHECK_STUDIES)} of them are in cache shard 0 and will be diffed")

mismatches = []
n_compared = 0
for study_id in CHECK_STUDIES:
    study_series = train_series_csv[train_series_csv["StudyInstanceUID"] == study_id].copy()
    # Header probe FIRST: *_series.csv carries no SeriesDescription /
    # SequenceName / ScanOptions / ScanningSequence / RepetitionTime /
    # EchoTime / Laterality, all of which the two calls below need.
    probed = probe_study_series(RAW_DIR, "train_series", study_id, study_series)
    annotated = annotate_series_headers(probed)
    chosen = select_slot_series(annotated, plane_map)
    laterality = resolve_study_laterality(probed)

    live_cache, live_mask = decode_study_slots(
        RAW_DIR, "train_series", study_id, chosen, laterality
    )

    cache_row = cache_row_of[study_id]
    # take_group(rows, g=1): the centre anchor's 3-slice slot, matching
    # config.GROUP_INDEX=1 (A2 v1's fixed training-time choice).
    ref = cache_arr[cache_row][:, 3:6]
    ref_mask = np.load(CACHE_DIR / "train.s00of04_mask.npy")[cache_row]

    if not np.array_equal(live_mask, ref_mask):
        mismatches.append((study_id, "mask differs", live_mask, ref_mask))
        continue

    present = live_mask.astype(bool)
    diff = np.abs(live_cache[present].astype(int) - ref[present].astype(int))
    max_diff = int(diff.max()) if diff.size else 0
    mean_diff = float(diff.mean()) if diff.size else 0.0
    print(f"  {study_id[:20]}...: lat={laterality}, present slots "
          f"max|diff|={max_diff}, mean|diff|={mean_diff:.2f}")
    n_compared += 1
    if max_diff > 5:  # small tolerance for resize/rounding, not an exact-bitwise requirement
        mismatches.append((study_id, f"max_diff={max_diff}", None, None))

print(f"\n{len(CHECK_STUDIES) - len(mismatches)} / {len(CHECK_STUDIES)} studies matched the pre-built cache")
assert n_compared >= 3, (
    f"only {n_compared}/{len(CHECK_STUDIES)} studies were actually compared against the cache "
    f"(CHECK_STUDIES is already restricted to shard 0, so this means the mask itself "
    f"diverged on most of them) - the gate is not meaningful, investigate before trusting it"
)
assert not mismatches, (
    f"Live decode diverged from the pre-built cache on {len(mismatches)} studies: {mismatches} - "
    f"DO NOT trust decode_study_slots() on hidden test data until this is fixed."
)
print("Validation gate PASSED - live decode reproduces the pre-built cache.")

## Section 3: `timm` availability check (no internet at scoring time)

Training notebooks `pip install -q timm`, which will fail once internet
is disabled for the real scored rerun. `pretrained=False` below avoids
needing to *download weights* (we load our own fine-tuned state_dict
regardless) - but the package itself still needs to import without
network access. This cell is the actual verification (see spec section
6): if it fails, an offline wheel Dataset or vendoring `timm`'s
`vit_small_patch14_dinov2` construction becomes necessary - not resolved
here, since it can only be checked on Kaggle.

In [ ]:
try:
    import timm
    print("timm already importable, no install needed:", timm.__version__)
except ImportError:
    import subprocess
    result = subprocess.run(["pip", "install", "-q", "timm"], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            "timm is not preinstalled AND pip install failed (expected if "
            "internet is disabled) - need an offline wheel Dataset before "
            "the real scored submission. See spec section 6.\\n"
            f"pip stderr: {result.stderr}"
        )
    import timm
    print("timm installed via pip (internet was available this run):", timm.__version__)

## Section 4: model - SlotAttentionModel, unchanged from A2 v1

Hand-kept copy of `src/model.py::masked_finding_attention` +
`SlotAttentionModel` (06045eb) - no architecture changes for A4.

In [ ]:
import torch.nn as nn


def masked_finding_attention(embeddings, mask, query, head_weight, head_bias):
    if not (mask.sum(dim=1) > 0).all():
        raise ValueError("masked_finding_attention: a row has 0 present slots")
    scores = torch.einsum("od,bsd->bos", query, embeddings) / (embeddings.shape[-1] ** 0.5)
    expanded_mask = mask.unsqueeze(1).expand(-1, query.shape[0], -1)
    scores = scores.masked_fill(expanded_mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    context = torch.einsum("bos,bsd->bod", weights, embeddings)
    logits = (context * head_weight.unsqueeze(0)).sum(-1) + head_bias
    if torch.isnan(logits).any() or torch.isinf(logits).any():
        raise RuntimeError("masked_finding_attention produced NaN/Inf logits")
    return logits


class SlotAttentionModel(nn.Module):
    def __init__(self, n_findings=len(FINDINGS), n_slots=len(SLOT_NAMES),
                 backbone_name="vit_small_patch14_dinov2.lvd142m", unfreeze_last=6,
                 pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained, num_classes=0, img_size=224,
        )
        embed_dim = self.backbone.num_features
        self.query = nn.Parameter(torch.randn(n_findings, embed_dim) * (embed_dim ** -0.5))
        self.heads = nn.Linear(embed_dim, n_findings)
        self.embed_dim = embed_dim
        self.n_findings = n_findings
        self.n_slots = n_slots

    def forward(self, slot_images, slot_mask):
        B, S, C, H, W = slot_images.shape
        flat = slot_images.view(B * S, C, H, W)
        embeddings = self.backbone(flat).view(B, S, self.embed_dim)
        return masked_finding_attention(
            embeddings, slot_mask, self.query, self.heads.weight, self.heads.bias
        )

## Section 5: predict the test set, ensemble 4 folds, write submission.csv

Same defensive pattern as Fase 5's retired `06` submission notebook
(the one that actually scored real LB 0.596): every row starts at the
constant 0.5 fallback, a per-study try/except means one bad study never
aborts the whole run, and the final asserts guard column order / row
count / uniqueness before writing.

The all-0.5 `submission.csv` is written **before** the decode loop and
overwritten after it, so a rerun that times out mid-loop (or in which
every study fails) still leaves a valid submission on disk rather than
none at all.

In [ ]:
test_csv = pd.read_csv(RAW_DIR / "test.csv")
test_series_csv = pd.read_csv(RAW_DIR / "test_series.csv")
sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv")
assert list(sample_submission.columns) == ["StudyInstanceUID"] + LABEL_COLS
print(f"test.csv: {len(test_csv)} studies")

plane_map = dict(zip(test_series_csv["SeriesInstanceUID"], test_series_csv["Anatomical_Plane"]))

# Write the all-0.5 fallback submission BEFORE decoding anything. The
# per-study try/except only helps if the file itself exists: a timeout
# partway through the loop below, or a total decode failure, would
# otherwise leave /kaggle/working with no submission.csv at all - the
# exact outcome the fallback-first design exists to prevent. Overwritten
# with real predictions at the end of this cell.
submission = sample_submission.set_index("StudyInstanceUID").copy()
submission.loc[:, :] = 0.5
submission.reset_index().to_csv("/kaggle/working/submission.csv", index=False)
print("Wrote the all-0.5 fallback /kaggle/working/submission.csv (will be overwritten)")

fold_predictions = {fold_id: {} for fold_id in range(CV_FOLDS)}
failures = {}
t0 = time.time()

models = []
for fold_id in range(CV_FOLDS):
    ckpt_path = CHECKPOINT_PATHS[fold_id]
    m = SlotAttentionModel(pretrained=False).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.eval()
    models.append(m)
print(f"Loaded {len(models)} fold checkpoints")

for study_id in test_csv["StudyInstanceUID"]:
    try:
        study_series = test_series_csv[test_series_csv["StudyInstanceUID"] == study_id].copy()
        # Same order as the Section 2 gate: probe headers, THEN classify
        # slots and resolve laterality once for the whole study.
        probed = probe_study_series(RAW_DIR, "test_series", study_id, study_series)
        annotated = annotate_series_headers(probed)
        chosen = select_slot_series(annotated, plane_map)
        laterality = resolve_study_laterality(probed)

        slots, mask = decode_study_slots(
            RAW_DIR, "test_series", study_id, chosen, laterality
        )
        images = torch.from_numpy(slots).float().unsqueeze(0).to(DEVICE) / 255.0
        mask_t = torch.from_numpy(mask).float().unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            for fold_id, m in enumerate(models):
                probs = torch.sigmoid(m(images, mask_t)).cpu().numpy()[0]
                fold_predictions[fold_id][study_id] = probs
    except Exception as e:
        failures[study_id] = repr(e)

elapsed = time.time() - t0
n_done = len(test_csv) - len(failures)
print(f"Decoded+predicted: {n_done} / {len(test_csv)} in {elapsed:.1f}s ({elapsed / max(len(test_csv), 1):.3f}s/study)")
print(f"Failures: {len(failures)}")
for sid, err in list(failures.items())[:20]:
    print(f"  {sid[:25]}...: {err}")

# Ensemble: mean of sigmoid probabilities across the 4 folds, per study.
ensembled = {}
for study_id in test_csv["StudyInstanceUID"]:
    per_fold = [fold_predictions[f][study_id] for f in range(CV_FOLDS) if study_id in fold_predictions[f]]
    if per_fold:
        ensembled[study_id] = np.mean(per_fold, axis=0)

# `submission` already exists from the top of this cell, every row at the
# 0.5 fallback. Guard the empty case: with no successful study at all,
# pd.DataFrame({}).T is a 0x0 frame and `.columns = FINDINGS` raises
# ValueError - which would abort before the final write and throw away
# the fallback rows we do have.
if ensembled:
    ensembled_df = pd.DataFrame(ensembled).T
    ensembled_df.columns = FINDINGS
    ensembled_df.index.name = "StudyInstanceUID"
    print(f"\nEnsembled predictions: {ensembled_df.shape}")

    rename = {finding: OFFICIAL_LABEL_COLUMNS[finding] for finding in FINDINGS}
    ensembled_official = ensembled_df.rename(columns=rename)
    covered = submission.index.intersection(ensembled_official.index)
    submission.loc[covered, LABEL_COLS] = ensembled_official.loc[covered, LABEL_COLS]
else:
    covered = submission.index[:0]
    print("\nNo study produced a prediction - every row stays at the constant 0.5 fallback")

n_fallback = len(submission) - len(covered)
print(f"Studies with a real prediction: {len(covered)} / {len(submission)}")
print(f"Studies falling back to constant 0.5: {n_fallback}")

submission = submission.reset_index()
assert list(submission.columns) == ["StudyInstanceUID"] + LABEL_COLS
assert len(submission) == len(test_csv), f"Expected {len(test_csv)} rows, got {len(submission)}"
assert submission["StudyInstanceUID"].is_unique

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("\nRewrote /kaggle/working/submission.csv with the real predictions")
print(submission.head())

## Real output (fill in after running on Kaggle)